# Paired bootstrap: mean difference fusion vs image-only

In [ ]:
import json
import os
import re
import statistics
from pathlib import Path

import numpy as np

_p = Path.cwd().resolve()
for _ in range(12):
    if (_p / "experiments").is_dir() and (_p / "data").is_dir():
        PROJECT_ROOT = _p
        break
    _p = _p.parent
else:
    PROJECT_ROOT = Path.cwd().resolve()
os.chdir(PROJECT_ROOT)


def load_test_macro_f1(exp_name: str) -> dict[int, float]:
    out: dict[int, float] = {}
    base = PROJECT_ROOT / "experiments" / exp_name / "metrics" / "experiments"
    for path in base.glob("seed_*_results.json"):
        m = re.match(r"seed_(\d+)_results\.json$", path.name)
        if not m:
            continue
        obj = json.loads(path.read_text(encoding="utf-8"))
        out[int(m.group(1))] = float(obj["test_metrics"]["test_macro_f1"])
    return out


fusion = load_test_macro_f1("phase3_robustness")
image = load_test_macro_f1("imageonly_robustness")
seeds = sorted(fusion.keys() & image.keys())
if fusion.keys() != image.keys():
    raise ValueError(f"seed mismatch: only fusion {sorted(set(fusion)-set(image))} only image {sorted(set(image)-set(fusion))}")

d_list = [fusion[s] - image[s] for s in seeds]
d = np.asarray(d_list, dtype=np.float64)
n = d.shape[0]
mean_d = float(np.mean(d))
std_d = float(np.std(d, ddof=1)) if n > 1 else float("nan")
cohen_d = mean_d / std_d if std_d else float("nan")
wins = int(np.sum(d > 0))
ties = int(np.sum(d == 0))
losses = int(np.sum(d < 0))

B = 10_000
rng_seed = 42
rng = np.random.default_rng(rng_seed)
idx = rng.integers(0, n, size=(B, n))
boot_means = d[idx].mean(axis=1)

ci_low, ci_high = np.percentile(boot_means, [2.5, 97.5])
one_sided_lb = float(np.percentile(boot_means, 5.0))
p_one_sided = (1 + int(np.sum(boot_means <= 0.0))) / (B + 1)

alpha = 0.05
print("n", n)
print("mean_diff", mean_d)
print("stdev_diff", std_d)
print("paired_Cohen_d", cohen_d)
print("wins_ties_losses", wins, ties, losses)
print("median_diff", float(np.median(d)))
print("B", B, "rng_seed", rng_seed)
print("ci95_mean_diff", float(ci_low), float(ci_high))
print("one_sided_95_lower_bound_mean_diff", one_sided_lb)
print("bootstrap_p_one_sided_mean_le_0", p_one_sided)
print("reject_H0_mean_le_0", p_one_sided < alpha)
print("one_sided_lb_gt_0", one_sided_lb > 0.0)